In [1]:
import numpy as np
import pandas as pd
import statistics
import seaborn as sns
import os
import subprocess
import shutil
import plotly.express as px
import sys
sys.path.append('/gpfs/commons/home/mgarbulowski/homic_package/src') 
from homic import file_readers, kraken2, process_data, make_plots
from collections import Counter
from scipy import stats
import matplotlib.pyplot as plt

homic package imported


In [2]:
# define paths
path = "/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/Metagenomes"
db_path = "/gpfs/commons/home/mgarbulowski/016_proj_shm/ref_dbs/kraken/human_hg38"

# samples id 
#all_samps = ["KP005", "KP012", "KP013", "KP016", "KP024", 
#"KP025", "KP026", "KP027", "KP029","KP033","KP035","KP038","KP040","KP046","KP047","KP048"] # batch 1 + 3

all_samps = ["KP003","KP004","KP005","KP008","KP010","KP011","KP012",
         "KP013","KP016","KP021","KP024","KP025","KP026","KP027","KP029",
         "KP033","KP035","KP037","KP038","KP040","KP041","KP046","KP047","KP048","KP049","KP052"] # all batches

## loading metadata
meta = pd.read_csv("/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/metadata_kp.csv")
meta = meta.loc[meta['id'].isin(all_samps)] # keep only the sequenced ones
print(meta)
# intersecting with previous work
path1 = "/gpfs/commons/home/mgarbulowski/homic_package/files/65sp.txt"

with open(path1, "r") as f:
    bl_65sp = [line.strip() for line in f if line.strip()]
bl_65sp = [s.replace('[','').replace(']','') for s in bl_65sp]

# genera
gsg = bl_65sp
gsg = set([s.split()[0] for s in gsg])

       id  age   bmi  height  muac  fatpercent  fatfreemas       hiv  female  \
2   KP003   24  24.0   158.0  26.0        36.0        39.0  NEGATIVE       1   
3   KP004   26  24.0   158.0  26.0        29.0        42.0  NEGATIVE       1   
4   KP005   22  23.0   175.0  30.0        16.0        60.0  NEGATIVE       0   
7   KP008   28  20.0   168.0  27.0        15.0        48.0  NEGATIVE       0   
9   KP010   24  22.0   162.0  25.0        16.0        49.0  NEGATIVE       0   
10  KP011   29  29.0   148.0  31.0        36.0        40.0  NEGATIVE       1   
11  KP012   34  20.0   154.0  23.0        26.0        35.0  NEGATIVE       1   
12  KP013   22  20.0   175.0  25.0        26.0        35.0  NEGATIVE       0   
15  KP016   50  21.0   170.0  26.0        23.0        46.0  NEGATIVE       0   
20  KP021   42  21.0   168.0  23.0        19.0        47.0  NEGATIVE       0   
23  KP024   30  27.0   144.0  27.0        39.0        34.0  NEGATIVE       1   
24  KP025   29  24.0   164.0  28.0      

In [7]:
path_blast = "/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/contigs_all_x3/blastn_refseq/"


samps_ids = "KP003"
df_out = process_data.read_n_clean_blastn(path_blast + "/"+ samps_ids + "_blastn_report.txt", db = "rs", top_hits = True, evalue = 1e-200, pident=0.99, drop_sp = True, drop_uncultured = True, drop_bacterium=True, drop_virus=True, best_unique = False)

In [9]:
print(df_out)

        contig_id                 subject_id  pident  length  evalue  \
49917  k141_97189         ref|NZ_CP011524.1|  96.573   24860     0.0   
15652  k141_31313  ref|NZ_JBBMFQ010000002.1|  98.315   21959     0.0   
43561  k141_84943  ref|NZ_JACOQI010000024.1|  98.258   19808     0.0   
40844  k141_79624         ref|NZ_CP060632.1|  99.635   18617     0.0   
43144  k141_84136     ref|NZ_FTRU01000007.1|  96.083   20245     0.0   
...           ...                        ...     ...     ...     ...   
49826  k141_97008         ref|NZ_CP085932.1|  95.970     397     0.0   
46410  k141_90512  ref|NZ_JAJFOJ010000001.1|  92.444     450     0.0   
27280  k141_53566  ref|NZ_CAJSZR010000001.1|  84.211     665     0.0   
1365   k141_10259  ref|NZ_JAJEPT010000014.1|  88.889     522     0.0   
33906   k141_6630  ref|NZ_JAJEPW010000053.1|  91.314     472     0.0   

       bitscore  score                                      subject_title  \
49917   41181.0  22300  Intestinimonas butyriciproducens s

In [12]:
print(Counter(df_out["species"]))



Counter({'Segatella sinensis': 2034, 'Segatella copri': 1592, 'Coprococcus comes': 1576, 'Mediterraneibacter faecis': 1560, 'Phocaeicola plebeius': 1550, 'Anaerobutyricum hallii': 1530, 'Blautia wexlerae': 1477, 'Phocaeicola dorei': 1312, 'Hominisplanchenecus faecis': 1246, 'Blautia obeum': 1181, 'Phocaeicola vulgatus': 1178, 'Dorea formicigenerans': 1176, 'Roseburia faecis': 1153, 'Coprococcus ammoniilyticus': 1138, 'Blautia fusiformis': 1070, 'Bacteroides xylanisolvens': 1031, 'Simiaoa sunii': 1018, 'Dorea longicatena': 1008, 'Caecibacteroides pullorum': 989, 'Collinsella aerofaciens': 986, 'Faecalicoccus pleomorphus': 985, 'Odoribacter splanchnicus': 951, 'Wujia chipingensis': 843, 'Evtepia gabavorous': 818, 'Bacteroides togonis': 777, 'Roseburia amylophila': 741, 'Leyella stercorea': 679, 'Dialister hominis': 552, 'Bacteroides caccae': 539, 'Vescimonas sanitatis': 536, 'Megamonas funiformis': 527, 'Agathobacter rectalis': 521, 'Waltera intestinalis': 508, 'Thermophilibacter provenc